# Phase 3 — Architecture 4: LegalBlock-GQA

In [1]:
%pip -q install "peft==0.17.1" "transformers==4.55.4" "accelerate==1.10.1" "tokenizers==0.21.4" "huggingface_hub==0.34.4" "cloudpickle==3.1.1" scikit-learn


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 92.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 82.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.5/561.5 kB 25.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.34.4 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.2

In [2]:
import os, sys, json, math, time, random, gc, zipfile, subprocess, cloudpickle, hashlib, shutil, ast, urllib.request
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint
from tokenizers import Tokenizer
from huggingface_hub import hf_hub_download, HfApi
from sklearn.metrics import f1_score, classification_report
from peft import LoraConfig, IA3Config, LoHaConfig, PromptTuningConfig, inject_adapter_in_model
from peft.tuners.prompt_tuning import PromptEmbedding

BASE = Path('/kaggle/input/datasets/harshvardhanvar/a4-p3')
OUT = Path('/kaggle/working/phase3_a4_peft')
HF_TOKEN = os.environ['HF_TOKEN']
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['OMP_NUM_THREADS'] = '2'
OUT.mkdir(parents=True, exist_ok=True)
REPO = 'Exploration-Lab/CS-781-Capstone'
C = json.loads((BASE/'config.json').read_text())
R = dict(base=str(BASE), out=str(OUT), seed=781, length=512, target=320,
         micro=2, accum=8, eval_batch=8, rank=16, prompt=16,
         eval_steps=200, patience=5, classes=13)
METHODS = {
    'full': dict(epochs=6, lr=2e-5),
    'partial': dict(epochs=8, lr=5e-5),
    'lora': dict(epochs=10, lr=2e-4),
    'lorma': dict(epochs=10, lr=2e-4),
    'ia3': dict(epochs=10, lr=5e-4),
    'loha': dict(epochs=10, lr=2e-4),
    'prompt': dict(epochs=12, lr=3e-3),
    'distill': dict(epochs=8, lr=4e-5),
}
random.seed(R['seed'])
np.random.seed(R['seed'])
torch.manual_seed(R['seed'])


## Dataset and sentence windows

In [3]:
revision = HfApi(token=HF_TOKEN).dataset_info(REPO).sha
files = ['Train/CL_train.json', 'Train/CL_dev.json', 'Train/IT_train.json', 'Train/IT_dev.json', 'Train/label_vocab.json', 'Test/CL_test.json', 'Test/IT_test.json']
raw = {}
for name in files:
    p = hf_hub_download(REPO, 'Phase_3_RR/' + name, repo_type='dataset', revision=revision, token=HF_TOKEN)
    raw[Path(name).stem] = json.loads(Path(p).read_text())
vocab = raw['label_vocab']
if isinstance(vocab, dict) and 'label2id' in vocab:
    vocab = vocab['label2id']
if isinstance(vocab, dict) and 'id2label' in vocab:
    vocab = vocab['id2label']
if isinstance(vocab, list):
    labels = vocab
elif all((str(k).isdigit() for k in vocab)):
    labels = [vocab[k] for k in sorted(vocab, key=lambda k: int(k))]
else:
    labels = sorted(vocab, key=lambda k: int(vocab[k]))
label2id = {s: i for i, s in enumerate(labels)}
R['classes'] = len(labels)

def documents(data, domain, labelled):
    if isinstance(data, dict) and 'id' in data and ('text' in data):
        rows = [data]
    elif isinstance(data, dict):
        rows = [dict(v, id=k) if isinstance(v, dict) else dict(id=k, text=v) for k, v in data.items()]
    else:
        rows = data
    docs = []
    for row in rows:
        text = row['text']
        y = row['labels'] if labelled else []
        y = [label2id[v] if isinstance(v, str) and v in label2id else int(v) for v in y]
        docs.append(dict(id=str(row['id']), text=text, y=y, domain=domain))
    return docs
DOCS = {}
for split in ['train', 'dev', 'test']:
    DOCS[split] = sum([documents(raw[f'{domain}_{split}'], d, split != 'test') for d, domain in enumerate(['CL', 'IT'])], [])
    ids = [d['id'] for d in DOCS[split]]
train_texts = {tuple(d['text']) for d in DOCS['train']}
tok = Tokenizer.from_file(str(BASE / 'tokenizer.json'))
eos = tok.token_to_id('<eos>')
pad = tok.token_to_id('<pad>')
R['pad'] = pad

def encode_docs(docs):
    rows = []
    for doc in docs:
        parts = [e.ids for e in tok.encode_batch(doc['text'])]
        for i, ids in enumerate(parts):
            if len(ids) > R['target']:
                half = R['target'] // 2
                ids = ids[:half] + ids[-(R['target'] - half):]
            previous = sum([p + [eos] for p in parts[max(0, i - 3):i]], [])
            budget = R['length'] - len(ids) - 2
            context = previous[-budget:] if budget else []
            x = context + [eos] + ids + [eos]
            rows.append(dict(x=x, start=len(context) + 1, end=len(x) - 1, y=doc['y'][i] if doc['y'] else -100, domain=doc['domain'], pos=i / max(1, len(parts) - 1), id=doc['id'], sentence=i))
    return rows
DATA = {s: encode_docs(ds) for s, ds in DOCS.items()}
with (OUT / 'data.pkl').open('wb') as f:
    cloudpickle.dump(dict(rows=DATA, docs=DOCS, labels=labels), f)
R['run_id'] = hashlib.sha256(json.dumps([C, {k: v for k, v in R.items() if k != 'run_id'}, METHODS, labels, revision], sort_keys=True).encode()).hexdigest()
LORMA_COMMIT = 'ffb3c1de29a7f2ab880dc504a2e473b5a4b63292'
LORMA_URL = f'https://raw.githubusercontent.com/Exploration-Lab/LoRMA/{LORMA_COMMIT}/NLU/run_glue_lorma_plus.py'
lorma_source = urllib.request.urlopen(LORMA_URL).read().decode()
(OUT / 'lorma_source.txt').write_text(lorma_source)
manifest = dict(lorma_commit=LORMA_COMMIT, peft_version='0.17.1', repo=REPO, revision=revision, config=C, settings=R, methods=METHODS, labels=labels, files=files, tokenizer_sha256=hashlib.sha256((BASE / 'tokenizer.json').read_bytes()).hexdigest())
(OUT / 'manifest.json').write_text(json.dumps(manifest, indent=2))
shutil.copyfile(BASE / 'tokenizer.json', OUT / 'tokenizer.json')
print(' | '.join((f'{s}: {len(ds)} documents, {len(DATA[s]):,} sentences' for s, ds in DOCS.items())))


CL_train.json: 0.00B [00:00, ?B/s]

CL_dev.json: 0.00B [00:00, ?B/s]

IT_train.json: 0.00B [00:00, ?B/s]

IT_dev.json: 0.00B [00:00, ?B/s]

label_vocab.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

CL_test.json: 0.00B [00:00, ?B/s]

IT_test.json: 0.00B [00:00, ?B/s]

train: 80 documents, 16,845 sentences | dev: 10 documents, 2,142 sentences | test: 10 documents, 2,197 sentences


## LegalBlock-GQA backbone

In [4]:
class RMSNorm(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d))
    def forward(self, x):
        return (x.float() * torch.rsqrt(x.float().square().mean(-1, keepdim=True) + 1e-6)).to(x.dtype) * self.weight

class Attention(nn.Module):
    def __init__(self, c, global_attn):
        super().__init__()
        self.h, self.kv, self.d = c['heads'], c['kv_heads'], c['dim'] // c['heads']
        self.window, self.global_attn = c['window'], global_attn
        self.qkv = nn.Linear(c['dim'], (self.h + 2*self.kv)*self.d, bias=False)
        self.out = nn.Linear(c['dim'], c['dim'], bias=False)
        self.register_buffer('freq', 1 / (10000 ** (torch.arange(0, self.d, 2).float()/self.d)), persistent=False)
    def rope(self, x, pos):
        angle = (torch.arange(pos, pos+x.size(2), device=x.device).float()[:, None] * self.freq).to(x.dtype)
        a, b = x[..., 0::2], x[..., 1::2]
        return torch.stack((a*angle.cos()-b*angle.sin(), a*angle.sin()+b*angle.cos()), -1).flatten(-2)
    def forward(self, x, cache=None, pos=0, use_cache=False):
        b, t, _ = x.shape
        q, k, v = self.qkv(x).split([self.h*self.d, self.kv*self.d, self.kv*self.d], -1)
        q = self.rope(q.view(b,t,self.h,self.d).transpose(1,2), pos)
        k = self.rope(k.view(b,t,self.kv,self.d).transpose(1,2), pos)
        v = v.view(b,t,self.kv,self.d).transpose(1,2)
        if cache is not None and (self.global_attn or pos % self.window != 0):
            k, v = torch.cat((cache[0], k), 2), torch.cat((cache[1], v), 2)
        saved = None
        if use_cache:
            start = 0 if self.global_attn else ((pos+t-1)//self.window)*self.window - pos
            start = max(0, start) if cache is None else 0
            saved = (k[:,:,start:], v[:,:,start:])
        k, v = k.repeat_interleave(self.h//self.kv, 1), v.repeat_interleave(self.h//self.kv, 1)
        if t == 1:
            y = F.scaled_dot_product_attention(q, k, v)
        elif self.global_attn:
            y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        else:
            w = self.window
            pad = (-t) % w
            q, k, v = [F.pad(z, (0,0,0,pad)).reshape(b,self.h,-1,w,self.d)
                       .permute(0,2,1,3,4).reshape(-1,self.h,w,self.d) for z in (q,k,v)]
            y = F.scaled_dot_product_attention(q,k,v,is_causal=True)
            y = y.reshape(b,-1,self.h,w,self.d).permute(0,2,1,3,4).reshape(b,self.h,-1,self.d)[:,:,:t]
        return self.out(y.transpose(1,2).reshape(b,t,-1)), saved

class Block(nn.Module):
    def __init__(self, c, i):
        super().__init__()
        self.n1, self.n2 = RMSNorm(c['dim']), RMSNorm(c['dim'])
        self.attn = Attention(c, (i+1) % c['global_every'] == 0)
        self.up = nn.Linear(c['dim'], 2*c['ff'], bias=False)
        self.down = nn.Linear(c['ff'], c['dim'], bias=False)
    def forward(self, x, cache=None, pos=0, use_cache=False):
        y, cache = self.attn(self.n1(x), cache, pos, use_cache)
        x = x + y
        a, b = self.up(self.n2(x)).chunk(2, -1)
        return x + self.down(F.silu(a)*b), cache

class LegalLM(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c = c
        self.emb = nn.Embedding(c['vocab'], c['dim'])
        self.blocks = nn.ModuleList([Block(c, i) for i in range(c['layers'])])
        self.norm = RMSNorm(c['dim'])
        self.apply(self.init_weights)
        for block in self.blocks:
            nn.init.normal_(block.attn.out.weight, std=.02/math.sqrt(2*c['layers']))
            nn.init.normal_(block.down.weight, std=.02/math.sqrt(2*c['layers']))
    def init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, std=.02)


In [5]:
tree = ast.parse((OUT / 'lorma_source.txt').read_text())
node = next((v for v in tree.body if isinstance(v, ast.ClassDef) and v.name == 'LormaLinear_plus'))
namespace = dict(torch=torch, nn=nn, F=F, math=math, __name__='__main__')
exec(compile(ast.Module(body=[node], type_ignores=[]), '<LORMA authors implementation>', 'exec'), namespace)
LormaLinear = namespace['LormaLinear_plus']

class SplitQKV(nn.Module):

    def __init__(self, base, c):
        super().__init__()
        d = c['dim']
        kv = c['kv_heads'] * (d // c['heads'])
        self.q_proj, self.k_proj, self.v_proj = [nn.Linear(d, k, bias=False) for k in [d, kv, kv]]
        with torch.no_grad():
            for layer, w in zip([self.q_proj, self.k_proj, self.v_proj], base.weight.split([d, kv, kv], 0)):
                layer.weight.copy_(w)

    def forward(self, x):
        return torch.cat((self.q_proj(x), self.k_proj(x), self.v_proj(x)), -1)

class RoleModel(nn.Module):

    def __init__(self, lm, method, r):
        super().__init__()
        self.lm = lm
        d = lm.c['dim']
        self.domain = nn.Embedding(2, 8)
        self.head = nn.Sequential(nn.Dropout(0.15), nn.Linear(2 * d + 9, 256), nn.GELU(), nn.Dropout(0.15), nn.Linear(256, r['classes']))
        self.prompt = None
        if method == 'prompt':
            config = PromptTuningConfig(num_virtual_tokens=r['prompt'], token_dim=d, num_transformer_submodules=1)
            self.prompt = PromptEmbedding(config, lm.emb)
            self.adapter_config = config

    def forward(self, x, start, end, domain, pos):
        h = self.lm.emb(x)
        shift = 0
        if self.prompt is not None:
            shift = self.prompt.embedding.num_embeddings
            tokens = torch.arange(shift, device=x.device)[None].expand(x.size(0), -1)
            h = torch.cat((self.prompt(tokens), h), 1)
        for block in self.lm.blocks:
            h, _ = checkpoint(block, h, use_reentrant=False) if self.training else block(h)
        h = self.lm.norm(h)
        start, end = (start + shift, end + shift)
        t = torch.arange(h.size(1), device=h.device)[None]
        mask = (t >= start[:, None]) & (t < end[:, None])
        mean = (h * mask[:, :, None]).sum(1) / mask.sum(1).clamp_min(1)[:, None]
        last = h[torch.arange(h.size(0), device=h.device), end]
        return self.head(torch.cat((mean, last, self.domain(domain), pos[:, None]), -1))

def make_model(method, c, r):
    saved = torch.load(Path(r['base']) / 'best.pt', map_location='cpu', weights_only=False)
    config = saved.get('config', c).copy()
    state = saved['model']
    if method == 'distill':
        chosen = np.linspace(2, config['layers'] - 1, 8, dtype=int).tolist()
        config['layers'] = 8
        state = {**{k: v for k, v in state.items() if not k.startswith('blocks.')}, **{f'blocks.{i}.' + k.split('.', 2)[2]: v for i, j in enumerate(chosen) for k, v in state.items() if k.startswith(f'blocks.{j}.')}}
    lm = LegalLM(config)
    lm.load_state_dict(state, strict=True)
    del saved, state
    for b in lm.blocks:
        b.attn.qkv = SplitQKV(b.attn.qkv, config)
    if method not in ['full', 'distill']:
        lm.requires_grad_(False)
    if method == 'partial':
        for b in lm.blocks[-6:]:
            b.requires_grad_(True)
        lm.norm.requires_grad_(True)
    targets = ['q_proj', 'v_proj', 'out', 'up', 'down']
    adapter_config = None
    if method == 'lora':
        adapter_config = LoraConfig(r=r['rank'], lora_alpha=2 * r['rank'], lora_dropout=0.05, target_modules=targets, bias='none')
    if method == 'ia3':
        adapter_config = IA3Config(target_modules=['k_proj', 'v_proj', 'down'], feedforward_modules=['down'])
    if method == 'loha':
        adapter_config = LoHaConfig(r=8, alpha=16, target_modules=['q_proj', 'v_proj', 'out'])
    if adapter_config is not None:
        lm = inject_adapter_in_model(adapter_config, lm)
    if method == 'lorma':
        for name, layer in list(lm.named_modules()):
            if isinstance(layer, nn.Linear) and name.split('.')[-1] in targets:
                parent, key = name.rsplit('.', 1)
                setattr(lm.get_submodule(parent), key, LormaLinear(layer, r['rank'], 2 * r['rank'], 'pre', True, 0.05))
    model = RoleModel(lm, method, r)
    if adapter_config is not None:
        model.adapter_config = adapter_config
    return model

def collate(rows, indices, device, r):
    selected = [rows[int(i)] for i in indices]
    length = math.ceil(max((len(v['x']) for v in selected)) / 8) * 8
    x = torch.full((len(selected), length), r['pad'], dtype=torch.long)
    for i, v in enumerate(selected):
        x[i, :len(v['x'])] = torch.tensor(v['x'])
    values = [x] + [torch.tensor([v[k] for v in selected], dtype=dtype) for k, dtype in [('start', torch.long), ('end', torch.long), ('domain', torch.long), ('pos', torch.float32), ('y', torch.long)]]
    return [v.pin_memory().to(device, non_blocking=True) for v in values]

@torch.inference_mode()
def predict(model, rows, device, r):
    model.eval()
    logits = np.empty((len(rows), r['classes']), dtype=np.float32)
    order = sorted(range(len(rows)), key=lambda i: len(rows[i]['x']))
    for at in range(0, len(order), r['eval_batch']):
        ix = order[at:at + r['eval_batch']]
        x, s, e, d, p, y = collate(rows, ix, device, r)
        with torch.autocast('cuda', dtype=torch.float16):
            z = model(x, s, e, d, p)
        logits[ix] = z.float().cpu().numpy()
    return logits

def probabilities(z):
    z = z - z.max(axis=1, keepdims=True)
    p = np.exp(z)
    return p / p.sum(axis=1, keepdims=True)

def write_submission(rows, docs, labels, logits, path):
    prediction = {d['id']: [None] * len(d['text']) for d in docs}
    for row, pred in zip(rows, logits.argmax(1)):
        prediction[row['id']][row['sentence']] = labels[int(pred)]
    payload = json.dumps(prediction, ensure_ascii=False, indent=2)
    with zipfile.ZipFile(path, 'w', zipfile.ZIP_DEFLATED) as z:
        z.writestr('prediction.json', payload)
    return hashlib.sha256(payload.encode()).hexdigest()


## Training

In [6]:
def run_method(gpu, method, c, r, methods, teacher=None):
    torch.set_num_threads(2)
    torch.cuda.set_device(gpu)
    device = torch.device('cuda', gpu)
    torch.manual_seed(r['seed'])
    np.random.seed(r['seed'])
    random.seed(r['seed'])
    root = Path(r['out'])
    folder = root / method
    folder.mkdir(exist_ok=True)
    if (folder / 'done.json').exists():
        previous = json.loads((folder / 'done.json').read_text())
        print(f'GPU {gpu} | {method} already complete; reusing outputs', flush=True)
        return
    with (root / 'data.pkl').open('rb') as f:
        data = cloudpickle.load(f)
    train, dev, test = [data['rows'][s] for s in ['train', 'dev', 'test']]
    gold = np.array([v['y'] for v in dev])
    soft = None
    if method == 'distill':
        tm = make_model(teacher, c, r)
        ck = torch.load(root / teacher / 'best.pt', map_location='cpu', weights_only=False)
        tm.load_state_dict(ck['weights'], strict=False)
        tm.to(device)
        soft = predict(tm, train, device, r)
        np.save(folder / 'teacher_logits.npy', soft)
        del tm, ck
        gc.collect()
        torch.cuda.empty_cache()
    model = make_model(method, c, r).to(device)
    if hasattr(model, 'adapter_config'):
        model.adapter_config.save_pretrained(folder)
    params = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
    body = [p for n, p in params if not n.startswith(('head.', 'domain.'))]
    head = [p for n, p in params if n.startswith(('head.', 'domain.'))]
    lr = methods[method]['lr']
    opt = torch.optim.AdamW([dict(params=body, lr=lr, initial_lr=lr), dict(params=head, lr=0.0003, initial_lr=0.0003)], weight_decay=0.01)
    scaler = torch.amp.GradScaler('cuda', init_scale=1024)
    count = np.bincount([v['y'] for v in train], minlength=r['classes'])
    weight = 1 / np.sqrt(np.maximum(count, 1))
    weight = np.clip(weight / weight.mean(), 0.25, 4.0)
    weight = torch.tensor(weight, dtype=torch.float32, device=device)
    nsteps = math.ceil(len(train) / (r['micro'] * r['accum']))
    total = methods[method]['epochs'] * nsteps
    warmup = max(10, int(0.06 * total))
    best, stale, step, history = (-1.0, 0, 0, [])
    start = time.monotonic()
    total_params = sum((p.numel() for p in model.parameters()))
    trained = sum((p.numel() for _, p in params))
    torch.cuda.reset_peak_memory_stats(device)
    print(f'{method} | GPU {gpu} ({torch.cuda.get_device_name(device)}) | model on {next(model.parameters()).device} | {total_params:,} parameters | {trained:,} trainable', flush=True)
    stop = False
    for epoch in range(1, methods[method]['epochs'] + 1):
        order = np.random.default_rng(r['seed'] + epoch).permutation(len(train))
        loss_sum, seen = (0.0, 0)
        for at in range(0, len(order), r['micro'] * r['accum']):
            model.train()
            opt.zero_grad(set_to_none=True)
            group = order[at:at + r['micro'] * r['accum']]
            factor = min(1.0, (step + 1) / warmup) * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(1.0, step / total))))
            for g in opt.param_groups:
                g['lr'] = g['initial_lr'] * factor
            for offset in range(0, len(group), r['micro']):
                ix = group[offset:offset + r['micro']]
                x, s, e, d, p, y = collate(train, ix, device, r)
                with torch.autocast('cuda', dtype=torch.float16):
                    z = model(x, s, e, d, p)
                    ce = (F.cross_entropy(z.float(), y, reduction='none') * weight[y]).mean()
                    loss = ce
                    if soft is not None:
                        target = torch.from_numpy(soft[ix]).to(device)
                        kd = F.kl_div(F.log_softmax(z.float() / 2.0, -1), F.softmax(target / 2.0, -1), reduction='batchmean') * 4.0
                        loss = 0.5 * ce + 0.5 * kd
                    scaled = loss * (len(ix) / len(group))
                scaler.scale(scaled).backward()
                loss_sum += loss.detach().item() * len(ix)
                seen += len(ix)
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for _, p in params], 1.0)
            scaler.step(opt)
            scaler.update()
            step += 1
            evaluate = step % r['eval_steps'] == 0 or at + len(group) >= len(order)
            if evaluate:
                logits = predict(model, dev, device, r)
                score = f1_score(gold, logits.argmax(1), labels=list(range(r['classes'])), average='macro', zero_division=0)
                improved = score > best
                stale = 0 if improved else stale + 1
                if improved:
                    best = float(score)
                    weights = {n: p.detach().cpu().float() for n, p in params}
                    torch.save(dict(weights=weights, method=method, config=model.lm.c, epoch=epoch, step=step, score=best, teacher=teacher), folder / 'best.tmp')
                    os.replace(folder / 'best.tmp', folder / 'best.pt')
                    np.save(folder / 'dev_logits.npy', logits)
                row = dict(epoch=epoch, step=step, loss=loss_sum / max(seen, 1), macro_f1=float(score))
                history.append(row)
                (folder / 'history.json').write_text(json.dumps(history, indent=2))
                print(f"{method} | epoch {epoch} step {step} | loss {row['loss']:.4f} | dev F1 {score:.4f}", flush=True)
                if stale >= r['patience']:
                    stop = True
                    break
        if stop:
            break
    ck = torch.load(folder / 'best.pt', map_location='cpu', weights_only=False)
    model.load_state_dict(ck['weights'], strict=False)
    test_logits = predict(model, test, device, r)
    np.save(folder / 'test_logits.npy', test_logits)
    sub = root / 'submissions'
    sub.mkdir(exist_ok=True)
    write_submission(test, data['docs']['test'], data['labels'], test_logits, sub / f'submission_A4_{method}.zip')
    dev_logits = np.load(folder / 'dev_logits.npy')
    pred = dev_logits.argmax(1)
    report = classification_report(gold, pred, labels=list(range(r['classes'])), target_names=data['labels'], zero_division=0, output_dict=True)
    (folder / 'classification_report.json').write_text(json.dumps(report, indent=2))
    result = dict(method=method, run_id=r['run_id'], macro_f1=best, parameters=total_params, trainable=trained, epoch=ck['epoch'], step=ck['step'], last_step=step, teacher=teacher, stopping_reason='patience' if stop else 'epoch_limit', minutes=(time.monotonic() - start) / 60, peak_gb=torch.cuda.max_memory_allocated(device) / 2 ** 30)
    for dom, name in enumerate(['CL', 'IT']):
        mask = np.array([v['domain'] == dom for v in dev])
        result[name + '_macro_f1'] = float(f1_score(gold[mask], pred[mask], labels=list(range(r['classes'])), average='macro', zero_division=0))
    (folder / 'done.json').write_text(json.dumps(result, indent=2))


## Two-GPU execution

In [7]:
def run_jobs(names, worker=run_method):
    pending, running, finished = (list(names), {}, set())
    first = ['full', 'partial', 'lora', 'lorma', 'ia3', 'loha']
    while pending or running:
        for gpu, (process, path, method) in list(running.items()):
            status = process.poll()
            if status is not None:
                if status != 0:
                    raise subprocess.CalledProcessError(status, process.args)
                path.unlink()
                del running[gpu]
                finished.add(method)
                print(f'GPU {gpu} finished {method}', flush=True)
        for gpu in range(2):
            if gpu in running:
                continue
            ready = [m for m in pending if m != 'distill' or set(first).issubset(finished)]
            if not ready:
                continue
            method = ready[0]
            teacher = None
            if method == 'distill':
                teacher = max(first, key=lambda m: json.loads((OUT / m / 'done.json').read_text())['macro_f1'])
            path = OUT / f'worker_{gpu}.pkl'
            with path.open('wb') as f:
                cloudpickle.dump((worker, (gpu, method, C, R, METHODS, teacher)), f)
            command = "import cloudpickle,sys; f=open(sys.argv[1],'rb'); fn,args=cloudpickle.load(f); f.close(); fn(*args)"
            process = subprocess.Popen([sys.executable, '-u', '-c', command, str(path)], env=os.environ.copy())
            running[gpu] = (process, path, method)
            pending.remove(method)
            print(f'GPU {gpu} starting {method} | PID {process.pid}', flush=True)
        if running:
            time.sleep(0.5)
run_jobs(list(METHODS))


GPU 0 starting full | PID 81
GPU 1 starting partial | PID 82
full | GPU 0 (Tesla T4) | model on cuda:0 | 262,033,117 parameters | 262,033,117 trainable
partial | GPU 1 (Tesla T4) | model on cuda:1 | 262,033,117 parameters | 58,017,757 trainable
partial | epoch 1 step 200 | loss 0.7109 | dev F1 0.1943
partial | epoch 1 step 400 | loss 0.6045 | dev F1 0.3044
full | epoch 1 step 200 | loss 0.6908 | dev F1 0.2342
partial | epoch 1 step 600 | loss 0.5417 | dev F1 0.3296
partial | epoch 1 step 800 | loss 0.4942 | dev F1 0.3862
full | epoch 1 step 400 | loss 0.5631 | dev F1 0.3523
partial | epoch 1 step 1000 | loss 0.4522 | dev F1 0.3852
partial | epoch 1 step 1053 | loss 0.4427 | dev F1 0.3677
partial | epoch 2 step 1200 | loss 0.1950 | dev F1 0.3676
full | epoch 1 step 600 | loss 0.4949 | dev F1 0.3543
partial | epoch 2 step 1400 | loss 0.1759 | dev F1 0.3798
partial | epoch 2 step 1600 | loss 0.1726 | dev F1 0.3936
partial | epoch 2 step 1800 | loss 0.1638 | dev F1 0.3618
full | epoch 1 st

In [8]:
teacher = json.loads((OUT/'distill'/'done.json').read_text())['teacher']
print('Distillation teacher:',teacher)


Distillation teacher: lora


## Ensemble and submissions

In [9]:
gold = np.array([v['y'] for v in DATA['dev']])
results = [json.loads((OUT/m/'done.json').read_text()) for m in METHODS]
ranked = sorted(results,key=lambda v:v['macro_f1'],reverse=True)
members = [v['method'] for v in ranked[:3]]
dev_probs = np.stack([probabilities(np.load(OUT/m/'dev_logits.npy')) for m in members])
test_probs = np.stack([probabilities(np.load(OUT/m/'test_logits.npy')) for m in members])
candidates = [np.ones(len(members))/len(members),
              np.array([max(v['macro_f1'],1e-6) for v in ranked[:3]])]
candidates[1] /= candidates[1].sum()
scores = [f1_score(gold,np.einsum('m,mnc->nc',w,dev_probs).argmax(1),
                   labels=list(range(len(labels))),average='macro',zero_division=0) for w in candidates]
w = candidates[int(np.argmax(scores))]
ensemble = np.einsum('m,mnc->nc',w,test_probs)
selection = dict(members=members,weights=w.tolist(),dev_macro_f1=float(max(scores)))
(OUT/'ensemble.json').write_text(json.dumps(selection,indent=2))
results.append(dict(method='ensemble',macro_f1=float(max(scores)),members=members))
SUB = OUT/'submissions'
SUB.mkdir(exist_ok=True)
fingerprints = {}
for method in list(METHODS)+['ensemble']:
    logits = ensemble if method=='ensemble' else np.load(OUT/method/'test_logits.npy')
    digest = write_submission(DATA['test'],DOCS['test'],labels,logits,SUB/f'submission_A4_{method}.zip')
    if digest in fingerprints:
        print(f'Identical predictions: {method} and {fingerprints[digest]}')
    fingerprints[digest] = method
(OUT/'results.json').write_text(json.dumps(results,indent=2))
for v in sorted(results,key=lambda v:v['macro_f1'],reverse=True):
    print(f'{v["method"]:10s} dev Macro-F1 {v["macro_f1"]:.4f}')
print(f'Nine submission ZIPs: {SUB}')


ensemble   dev Macro-F1 0.4190
lora       dev Macro-F1 0.4057
full       dev Macro-F1 0.3993
lorma      dev Macro-F1 0.3971
partial    dev Macro-F1 0.3936
loha       dev Macro-F1 0.3927
prompt     dev Macro-F1 0.3653
ia3        dev Macro-F1 0.3633
distill    dev Macro-F1 0.3123
Nine submission ZIPs: /kaggle/working/phase3_a4_peft/submissions


In [10]:
with zipfile.ZipFile(OUT/'all_submissions_A4.zip','w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(SUB.glob('*.zip')): z.write(p,p.name)
    for name in ['results.json','ensemble.json','manifest.json']: z.write(OUT/name,name)
print(OUT/'all_submissions_A4.zip')


/kaggle/working/phase3_a4_peft/all_submissions_A4.zip


## Outputs